# 03D — Julia M3 full multi-start reproduction

This notebook is the **Julia-side execution entry point** for M3.

It does not reimplement the Gaussian-mixture likelihood inside the notebook. Instead, it executes the already validated Julia implementation in:

`julia/scripts/fit_mixture.jl`

That script uses the local Julia project under `julia/`, performs the canonical full Phase J2 search, and writes the Julia results to `pdata/`.

## Canonical Phase J2 specification

- Model: M3 two-component Gaussian-mixture environmental null
- Classes: ALL, SG, IG
- Gauss–Hermite quadrature: 100 nodes
- Optimizer: `Optim.NelderMead()`
- Maximum iterations: 6000 per start
- Initial grid: 180 starts per class
- Total: 540 starts
- JIT warm-up is separated from the benchmark timing

The script writes four canonical outputs:

- `pdata/revision2026_julia_j2_mixture_diagnostics.csv`
- `pdata/revision2026_julia_j2_mixture_parameter_estimates.csv`
- `pdata/revision2026_julia_j2_mixture_model_comparison.csv`
- `pdata/revision2026_julia_j2_benchmark.csv`

## Data requirement

The canonical Julia script expects the authorized annual dataset at:

`data/M_1920_2023.csv`

This notebook intentionally does not substitute the public synthetic dataset for the canonical cross-language audit, because doing so would produce different M3 values.

## Kernel

Run this notebook with a Julia 1.12.x kernel. The canonical audit used Julia 1.12.7.


## 1. Locate the project root and verify required files

The notebook searches upward from the current working directory until it finds the expected `julia/` project and Phase J2 script.


In [2]:
using Dates

function find_project_root(start_dir::AbstractString = pwd())
    p = abspath(start_dir)

    for _ in 1:8
        project_toml = joinpath(p, "julia", "Project.toml")
        fit_script = joinpath(p, "julia", "scripts", "fit_mixture.jl")

        if isfile(project_toml) && isfile(fit_script)
            return p
        end

        parent = dirname(p)

        if parent == p
            break
        end

        p = parent
    end

    error(
        "Could not locate the IDMs project root. " *
        "Open this notebook from the project tree containing " *
        "julia/Project.toml and julia/scripts/fit_mixture.jl."
    )
end

ROOT = find_project_root()
JULIA_DIR = joinpath(ROOT, "julia")
FIT_SCRIPT = joinpath(JULIA_DIR, "scripts", "fit_mixture.jl")
TEST_SCRIPT = joinpath(JULIA_DIR, "test", "runtests.jl")
DATA_FILE = joinpath(ROOT, "data", "M_1920_2023.csv")
PDATA_DIR = joinpath(ROOT, "pdata")

println("Project root : ", ROOT)
println("Julia project: ", JULIA_DIR)
println("Julia version: ", VERSION)
println("Fit script   : ", FIT_SCRIPT)
println("Data file    : ", DATA_FILE)
println("pdata        : ", PDATA_DIR)

required_files = [
    joinpath(JULIA_DIR, "Project.toml"),
    joinpath(JULIA_DIR, "Manifest.toml"),
    joinpath(JULIA_DIR, "src", "IDMLikelihoods.jl"),
    joinpath(JULIA_DIR, "src", "GaussianMixture.jl"),
    FIT_SCRIPT,
    TEST_SCRIPT,
]

missing = filter(p -> !isfile(p), required_files)

if !isempty(missing)
    error(
        "Required Julia files are missing:\n" *
        join(missing, "\n")
    )
end

if !isfile(DATA_FILE)
    error(
        "Canonical Phase J2 requires data/M_1920_2023.csv. " *
        "The public synthetic file is not substituted automatically."
    )
end

mkpath(PDATA_DIR)

println("\nProject-structure audit: PASS")


Project root : .
Julia project: ./julia
Julia version: 1.12.7
Fit script   : ./julia/scripts/fit_mixture.jl
Data file    : ./data/M_1920_2023.csv
pdata        : ./pdata

Project-structure audit: PASS


## 2. Activate and instantiate the Julia environment

`Project.toml` and `Manifest.toml` in `julia/` define the reproducible environment.


In [3]:
using Pkg

Pkg.activate(JULIA_DIR)
Pkg.instantiate()

println("\nActive project:")
println(Base.active_project())

println("\nPackage status:")
Pkg.status()


  Activating project at `./julia`



Active project:
./julia/Project.toml

Package status:
Project julia v0.1.0
Status `./julia/Project.toml`
  [336ed68f] CSV v0.10.17
  [a93c6f00] DataFrames v1.8.2
  [442a2c76] FastGaussQuadrature v1.3.0
  [2ab3a3ac] LogExpFunctions v1.0.1
⌃ [429524aa] Optim v2.2.2
  [4c63d2b9] StatsFuns v2.2.1
  [8dfed614] Test v1.11.0
Info Packages marked with ⌃ have new versions available and may be upgradable.


## 3. Run the Julia validation suite

The current Julia package includes the Phase J1/J2 unit and integration tests. The canonical full audit reported 57 / 57 passing tests.

Set `RUN_TESTS = false` only if the same local Julia tree has already been tested and you want to proceed directly to the 540-start fit.


In [4]:
RUN_TESTS = true

if RUN_TESTS
    println("Running Julia package test suite...")
    Pkg.test()
    println("\nJulia test suite completed.")
else
    println("Julia test suite skipped by user setting.")
end


Running Julia package test suite...


     Testing julia
      Status `/private/var/folders/95/yxxqkhhx0xvclmmw0bqhv6z40000gn/T/jl_0EB8fz/Project.toml`
  [336ed68f] CSV v0.10.17
  [a93c6f00] DataFrames v1.8.2
  [442a2c76] FastGaussQuadrature v1.3.0
  [2ab3a3ac] LogExpFunctions v1.0.1
⌃ [429524aa] Optim v2.2.2
  [4c63d2b9] StatsFuns v2.2.1
  [e35ddc65] julia v0.1.0 `./julia`
  [8dfed614] Test v1.11.0
      Status `/private/var/folders/95/yxxqkhhx0xvclmmw0bqhv6z40000gn/T/jl_0EB8fz/Manifest.toml`
  [47edcb42] ADTypes v1.24.0
  [79e6a3ab] Adapt v4.7.0
⌃ [4fba245c] ArrayInterface v7.30.0
  [336ed68f] CSV v0.10.17
  [944b1d66] CodecZlib v0.7.9
  [34da2185] Compat v4.18.1
  [187b0558] ConstructionBase v1.6.0
  [a8cc5b0e] Crayons v4.2.0
  [9a962f9c] DataAPI v1.16.0
  [a93c6f00] DataFrames v1.8.2
  [864edb3b] DataStructures v0.19.6
  [e2d170a0] DataValueInterfaces v1.0.0
  [a0c0ee7d] DifferentiationInterface v0.7.21
  [ffbed154] DocStringExtensions v0.9.5
  [4e289a0a] EnumX v1.0.7
  [442a2c76] FastGaussQuadrature v1.3.0
  [48062228


Julia test suite completed.


Test Summary:                                   | Pass  Total  Time
IDMLikelihoods Test Suite (Phase J1 & Phase J2) |   57     57  1.8s
     Testing julia tests passed 


## 4. Execute the canonical Phase J2 540-start search

The implementation remains in `julia/scripts/fit_mixture.jl`.

`include` loads the script into the Julia kernel; the notebook then calls its canonical `run_full_phase_j2_reproduction()` entry point explicitly.

Expected scale:

- 180 starts for ALL
- 180 starts for SG
- 180 starts for IG
- 540 starts total

The script overwrites the four dedicated `revision2026_julia_j2_*.csv` files in `pdata/` with the results of this run.


In [5]:
# Load the canonical Phase J2 implementation.
include(FIT_SCRIPT)

println("\nStarting canonical Julia Phase J2 reproduction...")
println("This performs 180 starts/class × 3 classes = 540 starts.")

t0 = time()

phase_j2_ok = run_full_phase_j2_reproduction()

elapsed = time() - t0

@assert phase_j2_ok === true

println("\nPhase J2 script returned successfully.")
println(
    "Notebook-observed wall time: ",
    round(elapsed, digits=2),
    " s"
)



Starting canonical Julia Phase J2 reproduction...


  Activating project at `./julia`


This performs 180 starts/class × 3 classes = 540 starts.
PHASE J2: FULL 540-START MULTI-START OPTIMIZATION REPRODUCTION

[0] Explicit JIT Warm-up Run...
  JIT Warm-up completed in 0.0094s (Excluded from benchmarks).

--------------------------------------------------------------------------
CLASS: ALL | Executing 180 Multi-Starts...
--------------------------------------------------------------------------
Class ALL Summary:
  Successful Starts : 180 / 180
  Best Discovered NLL: 430.27129208471973 (Py Ref: 430.271292090558, Delta: -5.83827386435587e-9)
  Best Parameters    : mu1=-2.580053045826941, mu2=-2.4117626187975305, sig1=0.6970833071454813, sig2=0.37663392559899656, pi=0.19983741243550285
  2nd Mode NLL       : 430.28489552800505 (Delta vs Best: 0.013603443285319372)
  2nd Mode Params    : mu1=-2.4397632932873723, mu2=-1.821723123179826, sig1=0.39785821231488944, sig2=9.292513036621082e-6, pi=0.9698292440155724
  Class Execution Time: 17.25s (Iters: 63663, f_calls: 109625)

----

## 5. Verify the four output files

This checks that the current run produced the complete canonical Julia output set.


In [6]:
using CSV
using DataFrames

OUTPUT_FILES = [
    joinpath(PDATA_DIR, "revision2026_julia_j2_mixture_diagnostics.csv"),
    joinpath(PDATA_DIR, "revision2026_julia_j2_mixture_parameter_estimates.csv"),
    joinpath(PDATA_DIR, "revision2026_julia_j2_mixture_model_comparison.csv"),
    joinpath(PDATA_DIR, "revision2026_julia_j2_benchmark.csv"),
]

for p in OUTPUT_FILES
    if !isfile(p)
        error("Expected Phase J2 output was not created: $p")
    end

    st = stat(p)

    println(
        basename(p),
        " | bytes=",
        st.size,
        " | modified=",
        Dates.unix2datetime(st.mtime),
    )
end

println("\nPhase J2 output-file audit: PASS")


revision2026_julia_j2_mixture_diagnostics.csv | bytes=131285 | modified=2026-09-06T10:02:02.390
revision2026_julia_j2_mixture_parameter_estimates.csv | bytes=825 | modified=2026-09-06T10:02:02.481
revision2026_julia_j2_mixture_model_comparison.csv | bytes=679 | modified=2026-09-06T10:02:02.591
revision2026_julia_j2_benchmark.csv | bytes=754 | modified=2026-09-06T10:02:02.761

Phase J2 output-file audit: PASS


## 6. Benchmark and 540-start diagnostics audit

The benchmark CSV should contain exactly one row per class. The diagnostics CSV should contain 540 optimization starts in total.

The numerical reference below is used only as a regression audit of the canonical real-data run. It does not overwrite any result.


In [8]:
BENCHMARK_FILE = joinpath(
    PDATA_DIR,
    "revision2026_julia_j2_benchmark.csv",
)

DIAGNOSTICS_FILE = joinpath(
    PDATA_DIR,
    "revision2026_julia_j2_mixture_diagnostics.csv",
)

bench = CSV.read(
    BENCHMARK_FILE,
    DataFrame,
)

diag = CSV.read(
    DIAGNOSTICS_FILE,
    DataFrame,
)

println("=== Julia J2 benchmark ===")
display(bench)


required_bench = [
    "timestamp",
    "julia_version",
    "class",
    "model",
    "GH_N",
    "optimizer",
    "start_count",
    "successful_starts",
    "iterations",
    "evaluations",
    "opt_time_sec",
    "best_nll",
    "mu1_hat",
    "mu2_hat",
    "sigma1_hat",
    "sigma2_hat",
    "pi_hat",
]

missing_bench = setdiff(
    required_bench,
    names(bench),
)

@assert isempty(missing_bench)
@assert nrow(bench) == 3
@assert Set(String.(bench.class)) == Set(["ALL", "SG", "IG"])
@assert all(Int.(bench.GH_N) .== 100)
@assert all(String.(bench.optimizer) .== "NelderMead")
@assert all(Int.(bench.start_count) .== 180)
@assert all(Int.(bench.successful_starts) .== 180)

@assert nrow(diag) == 540

for cls in ["ALL", "SG", "IG"]
    sub = diag[String.(diag.class) .== cls, :]
    @assert nrow(sub) == 180
end

println("\nStructure / start-count audit: PASS")


=== Julia J2 benchmark ===


Row,timestamp,julia_version,class,model,GH_N,optimizer,start_count,successful_starts,iterations,evaluations,opt_time_sec,best_nll,mu1_hat,mu2_hat,sigma1_hat,sigma2_hat,pi_hat
,DateTime,String7,String3,String3,Int64,String15,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,2026-09-06T19:01:25,1.12.7,ALL,M3,100,NelderMead,180,180,63663,109625,17.2495,430.271,-2.58005,-2.41176,0.697083,0.376634,0.199837
2,2026-09-06T19:01:42,1.12.7,SG,M3,100,NelderMead,180,180,63222,107569,17.0635,416.177,-2.83723,-1.96269,0.895836,0.337446,0.161096
3,2026-09-06T19:02:01,1.12.7,IG,M3,100,NelderMead,180,180,76455,125947,18.9354,180.113,-3.41943,-2.60877,0.499542,3.2823e-5,0.883578



Structure / start-count audit: PASS


## 7. NLL regression audit

The canonical Julia Phase J2 best-found likelihoods are:

- ALL: 430.2712920847
- SG: 416.1772074557
- IG: 180.1129479483

A tolerance of `1e-6` is used here. Mixture parameters themselves are not hard-gated because near-degenerate components, especially IG, can have numerically different parameter representatives with essentially identical likelihood.


In [9]:
REFERENCE_NLL = Dict(
    "ALL" => 430.2712920847,
    "SG"  => 416.1772074557,
    "IG"  => 180.1129479483,
)

NLL_TOL = 1e-6

audit_rows = []

for row in eachrow(bench)
    cls = String(row.class)
    observed = Float64(row.best_nll)
    reference = REFERENCE_NLL[cls]
    delta = observed - reference
    passed = abs(delta) <= NLL_TOL

    push!(
        audit_rows,
        (
            class = cls,
            observed_best_nll = observed,
            reference_best_nll = reference,
            delta = delta,
            passed = passed,
        )
    )
end

nll_audit = DataFrame(audit_rows)

println("=== Julia J2 best-NLL regression audit ===")
display(nll_audit)

@assert all(nll_audit.passed)

println("\nBest-NLL regression audit: PASS")


=== Julia J2 best-NLL regression audit ===


Row,class,observed_best_nll,reference_best_nll,delta,passed
,String,Float64,Float64,Float64,Bool
1,ALL,430.271,430.271,1.97247e-11,true
2,SG,416.177,416.177,4.03588e-11,true
3,IG,180.113,180.113,2.84217e-14,true



Best-NLL regression audit: PASS


## 8. Inspect the nearest distinct mode in each class

This reproduces the canonical mode-diagnostics logic: after sorting by `final_nll`, the second mode is the first solution more than `1e-4` NLL above the best solution.


In [10]:
println("=== Julia J2 mode diagnostics ===")

mode_rows = []

for cls in ["ALL", "SG", "IG"]
    sub = diag[String.(diag.class) .== cls, :]

    finite_mask = isfinite.(Float64.(sub.final_nll))
    sub = sub[finite_mask, :]
    sort!(sub, :final_nll)

    @assert nrow(sub) > 0

    best_nll = Float64(sub.final_nll[1])

    second_idx = findfirst(
        x -> abs(Float64(x) - best_nll) > 1e-4,
        sub.final_nll,
    )

    second_nll = (
        isnothing(second_idx)
        ? missing
        : Float64(sub.final_nll[second_idx])
    )

    push!(
        mode_rows,
        (
            class = cls,
            best_nll = best_nll,
            second_mode_nll = second_nll,
            delta_second_vs_best = (
                ismissing(second_nll)
                ? missing
                : second_nll - best_nll
            ),
        )
    )
end

mode_audit = DataFrame(mode_rows)
display(mode_audit)


=== Julia J2 mode diagnostics ===


Row,class,best_nll,second_mode_nll,delta_second_vs_best
,String,Float64,Float64,Float64
1,ALL,430.271,430.285,0.0136034
2,SG,416.177,416.268,0.0910172
3,IG,180.113,180.376,0.263435


## 9. Handoff to the cross-language notebook

After this notebook and `03C_M3_python.ipynb` have both completed successfully, the next notebook should compare:

- Python: `pdata/m3_python_best_candidates.csv`
- Julia: `pdata/revision2026_julia_j2_benchmark.csv`
- Julia diagnostics: `pdata/revision2026_julia_j2_mixture_diagnostics.csv`

The next step should perform Python–Julia cross-evaluation and choose the **cross-language validated best-found candidate** for each class.

Do not describe the finite-mixture solution as a guaranteed global optimum.
